In [1]:
import bw2analyzer as ba
import bw2data as bd
import bw2calc as bc
import bw2io as bi
import matrix_utils as mu
import bw_processing as bp
import time
import wurst
import os
from pathlib import Path
import copy
import pandas as pd

In [2]:
bd.projects.set_current('Acetylene production')

In [3]:
bd.databases

Databases dictionary with 10 object(s):
	acetylene-production-Base-2020
	acetylene-production-Base-2050
	acetylene-production-RCP19-2050
	acetylene-production-RCP26-2050
	ecoinvent-3.10-biosphere
	ecoinvent-3.10-cutoff
	ecoinvent-3.10-cutoff-Base-2020
	ecoinvent-3.10-cutoff-Base-2050
	ecoinvent-3.10-cutoff-RCP19-2050
	ecoinvent-3.10-cutoff-RCP26-2050

In [4]:
db = bd.Database('acetylene-production-Base-2020')

In [5]:
method = [("IPCC 2021", "climate change", "GWP 100a, incl. H and bio CO2")]

In [6]:
def breakdown_calculations(db, activity):
    activities_list = [{activity : 1}]
    for exchange in activity.technosphere():
        activities_list.append({bd.Database(exchange.input.key[0]).get(exchange.input.key[1]) : exchange.amount})
    calculationSetup = {'inv' : activities_list, 'ia' : method}
    bd.calculation_setups['breakdown'] = calculationSetup
    my_lca = bc.MultiLCA('breakdown')
    results = pd.DataFrame(my_lca.results.transpose(), columns = [str(list(i.keys())[0]).split('\'')[1] for i in activities_list], index = pd.MultiIndex.from_tuples(method))
    results = results.sort_index().drop(index = [i for i in results.index if i[0] == 'rest'])
    direct_emissions = pd.DataFrame([0 if abs(results.iloc[r,0] - results.iloc[r,1:].sum()) / abs(results.iloc[r,0]) < 1e-5 else results.iloc[r, 0] - results.iloc[r, 1:].sum() for r in range(len(results.index))],
                                        columns = ['direct emissions'],
                                        index = results.index)
    results = pd.concat([results, direct_emissions], axis = 1)
    return results

In [7]:
df = pd.DataFrame()

In [8]:
activities = [activity for activity in db if 'vinyl chloride' in activity['reference product']]
activities

['vinyl chloride monomer production, fossil ethylene' (kilogram, GLO, None),
 'vinyl chloride monomer production, bio acetylene' (kilogram, GLO, None),
 'vinyl chloride monomer production, bio ethylene' (kilogram, GLO, None),
 'vinyl chloride monomer production, fossil acetylene' (kilogram, GLO, None)]

In [9]:
for activity in activities:
    df = pd.concat([df, breakdown_calculations(db, activity)], ignore_index = True)

In [10]:
df.fillna(0, inplace = True)
df

,"vinyl chloride monomer production, fossil ethylene","refrigeration, -50C",market for ethylene,"market for chlorine, gaseous","market for water, deionised","heat production, natural gas, at boiler condensing modulating >100kW","market group for electricity, high voltage",direct emissions,"vinyl chloride monomer production, bio acetylene","refrigeration, -25C","acetylene production, biochar, calcium carbide","market for hydrochloric acid, without water, in 30% solution state","vinyl chloride monomer production, bio ethylene",market for bio ethylene,"vinyl chloride monomer production, fossil acetylene","acetylene production, coal, calcium carbide"
0,2.34793,0.343214,1.110495,0.588774,0.101387,0.169762,0.016299,0.018,0.000000,0.000000,0.000000,0.000000,0.000000,0.000,0.000000,0.000000
1,0.00000,0.000000,0.000000,0.000000,0.039141,0.002710,0.000000,0.000,1.677215,0.052575,1.412812,0.169977,0.000000,0.000,0.000000,0.000000
2,0.00000,0.343214,0.000000,0.588774,0.101387,0.169762,0.016299,0.018,0.000000,0.000000,0.000000,0.000000,0.784435,-0.453,0.000000,0.000000
3,0.00000,0.000000,0.000000,0.000000,0.039141,0.002710,0.000000,0.000,0.000000,0.052575,0.000000,0.169977,0.000000,0.000,5.651539,5.387136


In [11]:
results_file_path = os.path.join('..', 'results', 'vcm_climate_change_impact_breakdown_2020.xlsx')
with pd.ExcelWriter(results_file_path, engine='xlsxwriter') as writer:
    df.to_excel(writer, sheet_name = 'overall')